In [ ]:
# CPU allocation / notebook configuration
# Edit CPU_LIST to a comma-separated list or range string such as "0-7,12".
import os

CPU_LIST = os.environ.get("NOTEBOOK_CPU_LIST", "")

def parse_cpu_list(text):
    cpus = []
    for part in str(text).split(','):
        part = part.strip()
        if not part:
            continue
        if '-' in part:
            left, right = part.split('-', 1)
            cpus.extend(range(int(left), int(right) + 1))
        else:
            cpus.append(int(part))
    return sorted(set(cpus))

_selected_cpus = parse_cpu_list(CPU_LIST)
if _selected_cpus and hasattr(os, 'sched_setaffinity'):
    os.sched_setaffinity(0, set(_selected_cpus))

for _name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_name, str(max(1, len(_selected_cpus) or 1)))

print("CPU affinity:", sorted(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else 'unavailable')


# GPU Alpha-Sweep Data Analysis

This notebook analyzes saved GPU campaign data under `gpu_data/`. It does not run dynamics; it only parses saved summaries and sample observables.

In [ ]:
from __future__ import annotations

import json
import math
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

try:
    mpl.rcParams.update({
        "font.family": "CMU Sans Serif",
        "mathtext.fontset": "cm",
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "axes.labelsize": 8,
        "axes.titlesize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "axes.linewidth": 0.8,
    })
except Exception:
    pass

ROOT = Path.cwd()
if ROOT.name != "class_A_fermionic_adaptive_circuit":
    # Allow execution from the notebook directory as well as repo root.
    candidate = ROOT.parent
    if (candidate / "colab_no_feedback_alpha_sweep_transfer").exists():
        ROOT = candidate

DATA_ROOT = ROOT / "colab_no_feedback_alpha_sweep_transfer" / "gpu_data"
OUTPUT_ROOT = ROOT / "colab_no_feedback_alpha_sweep_transfer" / "analysis_outputs" / "gpu_data_summary"
FIG_DIR = OUTPUT_ROOT / "figures"
TABLE_DIR = OUTPUT_ROOT / "tables"
for path in (OUTPUT_ROOT, FIG_DIR, TABLE_DIR):
    path.mkdir(parents=True, exist_ok=True)

FIG_WIDTH = 3.375
RUN_RE = re.compile(r"N(?P<Nx>\d+)x(?P<Ny>\d+)_dwtrunc(?P<dw>\d+)_corr_(?P<corr>none|perfect)_alpha_(?P<atag>.+)$")
EXPECTED_ALPHA = [1.0, 1.5, 1.8, 1.9, 2.0, 2.1, 2.2, 2.5, 3.0]
CORR_LABEL = {"none": "No feedback", "perfect": "Perfect correction"}
COLORS = {20: "#1f77b4", 30: "#2ca02c", 40: "#d62728"}
MARKERS = {"none": "o", "perfect": "s"}

print("DATA_ROOT", DATA_ROOT)
print("OUTPUT_ROOT", OUTPUT_ROOT)


In [ ]:
def parse_alpha_tag(tag: str) -> float:
    return float(str(tag).replace('p', '.'))


def load_json(path: Path):
    try:
        return json.loads(path.read_text())
    except Exception:
        return None


def nanmean_sem(values):
    arr = np.asarray(values, dtype=np.float64)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return np.nan, np.nan, 0
    mean = float(np.mean(finite))
    sem = float(np.std(finite, ddof=1) / math.sqrt(finite.size)) if finite.size > 1 else 0.0
    return mean, sem, int(finite.size)


def load_scalar(payload, key, default=np.nan):
    if key not in payload:
        return default
    value = payload[key]
    try:
        return value.item()
    except Exception:
        return value


def safe_float(value):
    try:
        return float(value)
    except Exception:
        return np.nan


def savefig(name: str):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    print(path)
    return path


In [ ]:
run_rows = []
sample_rows = []
finite_mode_rows = []
choi_rows = []
spectra_records = []
missing_or_bad = []

run_dirs = sorted(p for p in DATA_ROOT.rglob('runs/*') if p.is_dir() and (p / 'samples').is_dir())
for run_dir in run_dirs:
    match = RUN_RE.match(run_dir.name)
    if not match:
        missing_or_bad.append({"path": str(run_dir), "issue": "unparsed_run_name"})
        continue
    gd = match.groupdict()
    Nx = int(gd['Nx'])
    Ny = int(gd['Ny'])
    dwtrunc = int(gd['dw'])
    corr = gd['corr']
    alpha = parse_alpha_tag(gd['atag'])
    summary_path = run_dir / 'transfer_spectrum_summary.npz'
    run_summary_path = run_dir / 'run_summary.json'
    run_summary = load_json(run_summary_path) or {}
    sample_paths = sorted((run_dir / 'samples').glob('sample_*.npz'))
    summary_exists = summary_path.exists()

    summary_payload = {}
    if summary_exists:
        with np.load(summary_path, allow_pickle=False) as z:
            summary_payload = {k: z[k] for k in z.files}
            gaps = np.asarray(z['transfer_gap'], dtype=np.float64) if 'transfer_gap' in z else np.full(len(sample_paths), np.nan)
            finite_counts = np.asarray(z['finite_eigenstate_count'], dtype=np.float64) if 'finite_eigenstate_count' in z else np.full(len(sample_paths), np.nan)
            active = np.asarray(z['choi_active_final'], dtype=bool) if 'choi_active_final' in z else np.zeros(len(sample_paths), dtype=bool)
            min_abs_d = np.asarray(z['choi_min_abs_d'], dtype=np.float64) if 'choi_min_abs_d' in z else np.full(len(sample_paths), np.nan)
            basis_size = np.asarray(z['choi_basis_size'], dtype=np.int64) if 'choi_basis_size' in z else np.full(len(sample_paths), -1)
            exponents = np.asarray(z['transfer_exponents'], dtype=np.float64) if 'transfer_exponents' in z else None
            singular_values = np.asarray(z['transfer_singular_values'], dtype=np.float64) if 'transfer_singular_values' in z else None
    else:
        gaps = np.full(len(sample_paths), np.nan)
        finite_counts = np.full(len(sample_paths), np.nan)
        active = np.zeros(len(sample_paths), dtype=bool)
        min_abs_d = np.full(len(sample_paths), np.nan)
        basis_size = np.full(len(sample_paths), -1)
        exponents = None
        singular_values = None

    gap_mean, gap_sem, gap_finite_n = nanmean_sem(gaps)
    fc_mean, fc_sem, _ = nanmean_sem(finite_counts)
    min_abs_d_mean, min_abs_d_sem, _ = nanmean_sem(min_abs_d)
    active_frac = float(np.mean(active)) if active.size else np.nan
    run_rows.append({
        "run_name": run_dir.name,
        "run_dir": str(run_dir.relative_to(ROOT)),
        "Nx": Nx,
        "Ny": Ny,
        "dwtrunc": dwtrunc,
        "correction": corr,
        "alpha": alpha,
        "cycles": int(run_summary.get('cycles', 2 * Ny)),
        "sample_count": len(sample_paths),
        "summary_exists": bool(summary_exists),
        "gap_mean": gap_mean,
        "gap_sem": gap_sem,
        "gap_finite_sample_count": gap_finite_n,
        "gap_nan_count": int(np.count_nonzero(~np.isfinite(gaps))),
        "finite_mode_count_mean": fc_mean,
        "finite_mode_count_sem": fc_sem,
        "choi_active_fraction": active_frac,
        "choi_min_abs_d_mean": min_abs_d_mean,
        "choi_min_abs_d_sem": min_abs_d_sem,
        "choi_basis_size": int(basis_size[0]) if len(basis_size) else np.nan,
        "canonical_dynamics_entry_point": run_summary.get('canonical_dynamics_entry_point', ''),
        "gap_extraction": run_summary.get('gap_extraction', ''),
        "elapsed_s": safe_float(run_summary.get('elapsed_s', np.nan)),
    })

    if exponents is not None:
        mean_exp = np.nanmean(np.where(np.isfinite(exponents), exponents, np.nan), axis=0)
        finite_mask = np.isfinite(mean_exp)
        if np.any(finite_mask):
            finite_mean_exp = mean_exp[finite_mask]
            spectra_records.append({
                "Nx": Nx,
                "Ny": Ny,
                "correction": corr,
                "alpha": alpha,
                "kind": "transfer_exponents_mean",
                "index": np.arange(finite_mean_exp.size, dtype=np.int64),
                "value": finite_mean_exp.astype(np.float64),
            })
    if singular_values is not None:
        log_sv = np.log(np.clip(singular_values, 1e-300, 1e300))
        mean_log_sv = np.nanmean(np.where(np.isfinite(log_sv), log_sv, np.nan), axis=0)
        finite_mask = np.isfinite(mean_log_sv)
        if np.any(finite_mask):
            vals = mean_log_sv[finite_mask]
            spectra_records.append({
                "Nx": Nx,
                "Ny": Ny,
                "correction": corr,
                "alpha": alpha,
                "kind": "log_transfer_singular_values_mean",
                "index": np.arange(vals.size, dtype=np.int64),
                "value": vals.astype(np.float64),
            })

    for sample_path in sample_paths:
        try:
            with np.load(sample_path, allow_pickle=False) as z:
                sample_index = int(load_scalar(z, 'sample_index'))
                sample_seed = int(load_scalar(z, 'sample_seed'))
                sample_gap = safe_float(load_scalar(z, 'transfer_gap'))
                finite_count = int(load_scalar(z, 'finite_eigenstate_count', 0))
                active_final = bool(load_scalar(z, 'choi_active_final', False))
                failure_json = str(load_scalar(z, 'choi_failure_records_json', '[]'))
                try:
                    failures = json.loads(failure_json)
                except Exception:
                    failures = []
                finite_exps = np.asarray(z['finite_lyapunov_exponents'], dtype=np.float64) if 'finite_lyapunov_exponents' in z else np.array([], dtype=np.float64)
                finite_a = np.asarray(z['finite_lyapunov_a_eigenvalues'], dtype=np.float64) if 'finite_lyapunov_a_eigenvalues' in z else np.array([], dtype=np.float64)
                sample_rows.append({
                    "Nx": Nx,
                    "Ny": Ny,
                    "dwtrunc": dwtrunc,
                    "correction": corr,
                    "alpha": alpha,
                    "sample_index": sample_index,
                    "sample_seed": sample_seed,
                    "transfer_gap": sample_gap,
                    "finite_eigenstate_count": finite_count,
                    "finite_lyapunov_mode_count": int(load_scalar(z, 'finite_lyapunov_mode_count', finite_count)),
                    "finite_lyapunov_min_abs_exponent": float(np.min(np.abs(finite_exps))) if finite_exps.size else np.nan,
                    "finite_lyapunov_exponent_mean_abs": float(np.mean(np.abs(finite_exps))) if finite_exps.size else np.nan,
                    "particle_zero_count": int(load_scalar(z, 'particle_zero_count', 0)),
                    "particle_pole_count": int(load_scalar(z, 'particle_pole_count', 0)),
                    "a_eigenvalue_min": safe_float(load_scalar(z, 'a_eigenvalue_min')),
                    "a_eigenvalue_max": safe_float(load_scalar(z, 'a_eigenvalue_max')),
                    "choi_min_abs_d": safe_float(load_scalar(z, 'choi_min_abs_d')),
                    "choi_active_final": active_final,
                    "choi_basis_size": int(load_scalar(z, 'choi_basis_size', -1)),
                    "failure_count": len(failures),
                    "sample_path": str(sample_path.relative_to(ROOT)),
                })
                choi_rows.append({
                    "Nx": Nx,
                    "Ny": Ny,
                    "correction": corr,
                    "alpha": alpha,
                    "sample_index": sample_index,
                    "choi_active_final": active_final,
                    "choi_min_abs_d": safe_float(load_scalar(z, 'choi_min_abs_d')),
                    "failure_count": len(failures),
                    "failure_records_json": failure_json,
                })
                for mode_pos, value in enumerate(finite_exps.tolist()):
                    finite_mode_rows.append({
                        "Nx": Nx,
                        "Ny": Ny,
                        "correction": corr,
                        "alpha": alpha,
                        "sample_index": sample_index,
                        "mode_position": mode_pos,
                        "finite_lyapunov_exponent": float(value),
                        "abs_finite_lyapunov_exponent": abs(float(value)),
                        "finite_lyapunov_a_eigenvalue": float(finite_a[mode_pos]) if mode_pos < finite_a.size else np.nan,
                    })
        except Exception as exc:
            missing_or_bad.append({"path": str(sample_path), "issue": repr(exc)})

runs_df = pd.DataFrame(run_rows).sort_values(["Ny", "correction", "alpha"]).reset_index(drop=True)
samples_df = pd.DataFrame(sample_rows).sort_values(["Ny", "correction", "alpha", "sample_index"]).reset_index(drop=True)
finite_modes_df = pd.DataFrame(finite_mode_rows)
choi_health_df = pd.DataFrame(choi_rows)
gap_df = samples_df[["Nx", "Ny", "dwtrunc", "correction", "alpha", "sample_index", "transfer_gap"]].copy()

spectra_rows = []
for record in spectra_records:
    for idx, val in zip(record["index"], record["value"]):
        spectra_rows.append({
            "Nx": record["Nx"],
            "Ny": record["Ny"],
            "correction": record["correction"],
            "alpha": record["alpha"],
            "kind": record["kind"],
            "spectrum_index": int(idx),
            "value": float(val),
        })
spectra_df = pd.DataFrame(spectra_rows)
missing_df = pd.DataFrame(missing_or_bad)

print("runs", len(runs_df), "samples", len(samples_df), "finite mode rows", len(finite_modes_df), "spectra rows", len(spectra_df))
display(runs_df.head())


In [ ]:
# Save tidy tables.
runs_df.to_csv(TABLE_DIR / "runs_df.csv", index=False)
samples_df.to_csv(TABLE_DIR / "samples_df.csv", index=False)
gap_df.to_csv(TABLE_DIR / "gap_df.csv", index=False)
spectra_df.to_csv(TABLE_DIR / "spectra_df.csv", index=False)
finite_modes_df.to_csv(TABLE_DIR / "finite_modes_df.csv", index=False)
choi_health_df.to_csv(TABLE_DIR / "choi_health_df.csv", index=False)
missing_df.to_csv(TABLE_DIR / "missing_or_bad_df.csv", index=False)

np.savez_compressed(
    OUTPUT_ROOT / "summary_tables.npz",
    runs=runs_df.to_records(index=False),
    samples=samples_df.to_records(index=False),
)

completion_rows = []
for Ny in sorted(runs_df["Ny"].unique()):
    for corr in ["none", "perfect"]:
        have = sorted(runs_df[(runs_df.Ny == Ny) & (runs_df.correction == corr)]["alpha"].tolist())
        missing = [a for a in EXPECTED_ALPHA if a not in have]
        completion_rows.append({"Ny": Ny, "correction": corr, "alpha_count": len(have), "alphas": have, "missing_alphas": missing})
completion_df = pd.DataFrame(completion_rows)
completion_df.to_csv(TABLE_DIR / "completion_df.csv", index=False)
display(completion_df)


In [ ]:
# Build high-level summary tables.
run_summary_df = runs_df.copy()
smallest_gap_df = (
    runs_df[np.isfinite(runs_df["gap_mean"])]
    .sort_values(["Ny", "correction", "gap_mean"])
    .groupby(["Ny", "correction"], as_index=False)
    .first()[["Ny", "correction", "alpha", "gap_mean", "gap_sem", "finite_mode_count_mean", "choi_active_fraction", "choi_basis_size"]]
)
choi_summary_df = (
    runs_df.groupby(["Ny", "correction"], as_index=False)
    .agg(
        mean_active_fraction=("choi_active_fraction", "mean"),
        min_active_fraction=("choi_active_fraction", "min"),
        min_choi_min_abs_d=("choi_min_abs_d_mean", "min"),
        total_gap_nan_count=("gap_nan_count", "sum"),
        run_count=("run_name", "count"),
    )
)
finite_count_summary_df = (
    runs_df.groupby(["Ny", "correction", "alpha"], as_index=False)
    .agg(finite_mode_count_mean=("finite_mode_count_mean", "mean"), finite_mode_count_sem=("finite_mode_count_sem", "mean"))
)

smallest_gap_df.to_csv(TABLE_DIR / "smallest_gap_by_size_correction.csv", index=False)
choi_summary_df.to_csv(TABLE_DIR / "choi_summary.csv", index=False)
finite_count_summary_df.to_csv(TABLE_DIR / "finite_count_summary.csv", index=False)
run_summary_df.to_csv(TABLE_DIR / "run_summary_metrics.csv", index=False)

display(smallest_gap_df)
display(choi_summary_df)


In [ ]:
# Figure: data completeness heatmap.
fig, ax = plt.subplots(figsize=(FIG_WIDTH, 2.5))
heat_rows = []
for Ny in sorted(runs_df.Ny.unique()):
    for corr in ["none", "perfect"]:
        present = [int(((runs_df.Ny == Ny) & (runs_df.correction == corr) & np.isclose(runs_df.alpha, alpha)).any()) for alpha in EXPECTED_ALPHA]
        heat_rows.append((Ny, corr, present))
heat = np.asarray([row[2] for row in heat_rows], dtype=float)
im = ax.imshow(heat, aspect='auto', vmin=0, vmax=1, cmap='Greens')
ax.set_xticks(range(len(EXPECTED_ALPHA)), [f"{a:g}" for a in EXPECTED_ALPHA], rotation=45, ha='right')
ax.set_yticks(range(len(heat_rows)), [f"Ny={ny}, {CORR_LABEL[corr]}" for ny, corr, _ in heat_rows])
ax.set_xlabel(r"$\alpha_1$")
ax.set_title("Run completeness")
for y in range(heat.shape[0]):
    for x in range(heat.shape[1]):
        ax.text(x, y, "1" if heat[y, x] else "0", ha='center', va='center', fontsize=7)
savefig("completeness_heatmap.png")
plt.show()


In [ ]:
# Figure: transfer gap vs alpha.
fig, axes = plt.subplots(1, 2, figsize=(2 * FIG_WIDTH, 2.6), sharey=True)
for ax, corr in zip(axes, ["none", "perfect"]):
    sub_corr = runs_df[runs_df.correction == corr]
    for Ny, sub in sub_corr.groupby("Ny"):
        sub = sub.sort_values("alpha")
        ax.errorbar(sub.alpha, sub.gap_mean, yerr=sub.gap_sem, marker=MARKERS[corr], color=COLORS.get(Ny), label=f"Ny={Ny}", lw=1.0, ms=3)
    ax.set_title(CORR_LABEL[corr])
    ax.set_xlabel(r"$\alpha_1$")
    ax.set_yscale("log")
    ax.grid(True, alpha=0.25)
axes[0].set_ylabel("transfer gap")
axes[1].legend(frameon=False)
savefig("transfer_gap_vs_alpha.png")
plt.show()


In [ ]:
# Figure: no-feedback vs perfect correction gap comparison where both exist.
fig, ax = plt.subplots(figsize=(FIG_WIDTH, 2.7))
merged = runs_df.pivot_table(index=["Ny", "alpha"], columns="correction", values="gap_mean", aggfunc="first").reset_index()
merged = merged[np.isfinite(merged.get("none", np.nan)) & np.isfinite(merged.get("perfect", np.nan))]
for Ny, sub in merged.groupby("Ny"):
    sub = sub.sort_values("alpha")
    ax.plot(sub.alpha, sub.perfect / sub.none, marker="o", color=COLORS.get(Ny), label=f"Ny={Ny}", lw=1.0, ms=3)
ax.axhline(1.0, color="0.3", lw=0.8, ls="--")
ax.set_yscale("log")
ax.set_xlabel(r"$\alpha_1$")
ax.set_ylabel("perfect / no-feedback gap")
ax.set_title("Gap enhancement ratio")
ax.grid(True, alpha=0.25)
ax.legend(frameon=False)
savefig("gap_ratio_perfect_over_none.png")
plt.show()


In [ ]:
# Figure: finite Lyapunov mode count vs alpha.
fig, axes = plt.subplots(1, 2, figsize=(2 * FIG_WIDTH, 2.6), sharey=True)
for ax, corr in zip(axes, ["none", "perfect"]):
    for Ny, sub in runs_df[runs_df.correction == corr].groupby("Ny"):
        sub = sub.sort_values("alpha")
        ax.errorbar(sub.alpha, sub.finite_mode_count_mean, yerr=sub.finite_mode_count_sem, marker=MARKERS[corr], color=COLORS.get(Ny), label=f"Ny={Ny}", lw=1.0, ms=3)
    ax.set_title(CORR_LABEL[corr])
    ax.set_xlabel(r"$\alpha_1$")
    ax.grid(True, alpha=0.25)
axes[0].set_ylabel("finite mode count")
axes[1].legend(frameon=False)
savefig("finite_mode_count_vs_alpha.png")
plt.show()


In [ ]:
# Figure: Choi active fraction vs alpha.
fig, axes = plt.subplots(1, 2, figsize=(2 * FIG_WIDTH, 2.6), sharey=True)
for ax, corr in zip(axes, ["none", "perfect"]):
    for Ny, sub in runs_df[runs_df.correction == corr].groupby("Ny"):
        sub = sub.sort_values("alpha")
        ax.plot(sub.alpha, sub.choi_active_fraction, marker=MARKERS[corr], color=COLORS.get(Ny), label=f"Ny={Ny}", lw=1.0, ms=3)
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(CORR_LABEL[corr])
    ax.set_xlabel(r"$\alpha_1$")
    ax.grid(True, alpha=0.25)
axes[0].set_ylabel("Choi active fraction")
axes[1].legend(frameon=False)
savefig("choi_active_fraction_vs_alpha.png")
plt.show()


In [ ]:
# Figure: Choi min abs d vs alpha.
fig, axes = plt.subplots(1, 2, figsize=(2 * FIG_WIDTH, 2.6), sharey=True)
for ax, corr in zip(axes, ["none", "perfect"]):
    for Ny, sub in runs_df[runs_df.correction == corr].groupby("Ny"):
        sub = sub.sort_values("alpha")
        ax.errorbar(sub.alpha, sub.choi_min_abs_d_mean, yerr=sub.choi_min_abs_d_sem, marker=MARKERS[corr], color=COLORS.get(Ny), label=f"Ny={Ny}", lw=1.0, ms=3)
    ax.set_yscale("log")
    ax.set_title(CORR_LABEL[corr])
    ax.set_xlabel(r"$\alpha_1$")
    ax.grid(True, alpha=0.25)
axes[0].set_ylabel(r"mean min $|d|$")
axes[1].legend(frameon=False)
savefig("choi_min_abs_d_vs_alpha.png")
plt.show()


In [ ]:
# Figure: mean transfer exponent spectra for representative alphas.
representative_alphas = [1.0, 1.8, 2.0, 2.2]
fig, axes = plt.subplots(2, 2, figsize=(2 * FIG_WIDTH, 4.8), sharex=False, sharey=True)
for ax, alpha in zip(axes.ravel(), representative_alphas):
    sub = spectra_df[(spectra_df.kind == "transfer_exponents_mean") & np.isclose(spectra_df.alpha, alpha)]
    for (Ny, corr), group in sub.groupby(["Ny", "correction"]):
        group = group.sort_values("spectrum_index")
        if group.empty:
            continue
        ax.plot(group.spectrum_index, group.value, color=COLORS.get(Ny), ls="-" if corr == "none" else "--", lw=0.8, label=f"Ny={Ny}, {corr}")
    ax.axhline(0.0, color="0.3", lw=0.6)
    ax.set_title(rf"$\alpha_1={alpha:g}$")
    ax.set_xlabel("ordered finite index")
    ax.grid(True, alpha=0.2)
axes[0,0].set_ylabel("mean exponent")
axes[1,0].set_ylabel("mean exponent")
axes[0,1].legend(frameon=False, fontsize=6, loc="best")
savefig("mean_transfer_exponent_spectra_representative_alpha.png")
plt.show()


In [ ]:
# Figure: log singular-value spectra for representative alphas.
representative_alphas = [1.0, 1.8, 2.0, 2.2]
fig, axes = plt.subplots(2, 2, figsize=(2 * FIG_WIDTH, 4.8), sharex=False, sharey=True)
for ax, alpha in zip(axes.ravel(), representative_alphas):
    sub = spectra_df[(spectra_df.kind == "log_transfer_singular_values_mean") & np.isclose(spectra_df.alpha, alpha)]
    for (Ny, corr), group in sub.groupby(["Ny", "correction"]):
        group = group.sort_values("spectrum_index")
        if group.empty:
            continue
        ax.plot(group.spectrum_index, group.value, color=COLORS.get(Ny), ls="-" if corr == "none" else "--", lw=0.8, label=f"Ny={Ny}, {corr}")
    ax.axhline(0.0, color="0.3", lw=0.6)
    ax.set_title(rf"$\alpha_1={alpha:g}$")
    ax.set_xlabel("ordered finite index")
    ax.grid(True, alpha=0.2)
axes[0,0].set_ylabel("mean log singular value")
axes[1,0].set_ylabel("mean log singular value")
axes[0,1].legend(frameon=False, fontsize=6, loc="best")
savefig("mean_log_singular_spectra_representative_alpha.png")
plt.show()


In [ ]:
# Figure: finite Lyapunov exponent distribution.
fig, axes = plt.subplots(1, 2, figsize=(2 * FIG_WIDTH, 2.6), sharey=True)
for ax, corr in zip(axes, ["none", "perfect"]):
    sub = finite_modes_df[finite_modes_df.correction == corr]
    bins = np.linspace(0, min(0.25, np.nanpercentile(sub.abs_finite_lyapunov_exponent, 99) if len(sub) else 0.25), 50)
    for Ny, group in sub.groupby("Ny"):
        vals = group.abs_finite_lyapunov_exponent.to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        if vals.size:
            ax.hist(vals, bins=bins, histtype="step", density=True, color=COLORS.get(Ny), label=f"Ny={Ny}", lw=1.0)
    ax.set_title(CORR_LABEL[corr])
    ax.set_xlabel(r"$|\lambda|$")
    ax.grid(True, alpha=0.2)
axes[0].set_ylabel("density")
axes[1].legend(frameon=False)
savefig("finite_lyapunov_abs_exponent_distribution.png")
plt.show()


In [ ]:
# Write executive summary JSON.
missing_completion = completion_df[completion_df["missing_alphas"].map(len) > 0].copy()
summary_payload = {
    "data_root": str(DATA_ROOT.relative_to(ROOT)),
    "output_root": str(OUTPUT_ROOT.relative_to(ROOT)),
    "run_count": int(len(runs_df)),
    "sample_count": int(len(samples_df)),
    "npz_count": int(len(list(DATA_ROOT.rglob('*.npz')))),
    "json_count": int(len(list(DATA_ROOT.rglob('*.json')))),
    "csv_count": int(len(list(DATA_ROOT.rglob('*.csv')))),
    "completion": completion_df.to_dict(orient="records"),
    "missing_completion": missing_completion.to_dict(orient="records"),
    "smallest_gap_by_size_correction": smallest_gap_df.to_dict(orient="records"),
    "choi_summary": choi_summary_df.to_dict(orient="records"),
    "table_paths": sorted(str(p.relative_to(ROOT)) for p in TABLE_DIR.glob('*')),
    "figure_paths": sorted(str(p.relative_to(ROOT)) for p in FIG_DIR.glob('*.png')),
}
(OUTPUT_ROOT / "executive_summary.json").write_text(json.dumps(summary_payload, indent=2, sort_keys=True))
print(json.dumps(summary_payload, indent=2, sort_keys=True)[:6000])
